In [3]:
pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install XlsxWriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


All samples

In [15]:
# ======================================
# FULL PIPELINE WITH SUBJECT-WISE CV, BASELINE, AND SHAP
# ======================================

import os
import random
import copy
import time

import numpy as np
import pandas as pd
import librosa
import pywt
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

from collections import Counter

from sklearn.model_selection import GroupKFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import shap  # make sure shap is installed: pip install shap

sns.set_style("whitegrid")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ======================================
# 1. DATASET & FEATURE EXTRACTION
# ======================================

class SMSATDataset(Dataset):
    """
    Dataset for SMSAT hierarchy:
    root/
      Music/Generated-data/*.wav
      Normal(Silence)/Generated-data/*.wav
      SpiritualMeditation/Generated-data/*.wav

    Also extracts a subject ID from filename prefix before first '_'
    for subject-wise splitting (to avoid data leakage).
    """
    def __init__(self, root_dir, samples_per_class=30):
        self.root_dir = root_dir
        self.file_paths = []
        self.labels = []
        self.subject_ids = []   # for subject-wise splitting
        self.class_counts = Counter()

        self.class_mapping = {
            "Music": 0,
            "Normal(Silence)": 1,
            "SpiritualMeditation": 2
        }

        for category in self.class_mapping.keys():
            gen_path = os.path.join(root_dir, category, "Generated-data")
            if not os.path.isdir(gen_path):
                print(f"Warning: Missing folder {gen_path}")
                continue

            wav_files = [f for f in os.listdir(gen_path) if f.endswith(".wav")]

            # Limit samples per class for speed (adjust as needed)
            if len(wav_files) > samples_per_class:
                wav_files = random.sample(wav_files, samples_per_class)

            label = self.class_mapping[category]

            for f in wav_files:
                full_path = os.path.join(gen_path, f)
                self.file_paths.append(full_path)
                self.labels.append(label)

                # SUBJECT ID from filename (before first '_')
                base = os.path.splitext(f)[0]
                subject_id = base.split("_")[0]
                self.subject_ids.append(subject_id)

                self.class_counts[label] += 1

        print("Class counts:", self.class_counts)

    def __len__(self):
        return len(self.file_paths)

    def extract_features_from_path(self, file_path):
        waveform, sr = librosa.load(file_path, sr=16000)

        # MFCC (13)
        mfccs = librosa.feature.mfcc(y=waveform, sr=sr, n_mfcc=13).mean(axis=1)

        # Time features (2)
        zcr = librosa.feature.zero_crossing_rate(waveform)[0].mean()
        rms = librosa.feature.rms(y=waveform)[0].mean()
        time_features = np.array([zcr, rms])

        # Wavelet (10)
        coeffs = pywt.wavedec(waveform, 'db4', level=4)
        wavelet_features = []
        for c in coeffs:
            wavelet_features.append(np.mean(c) if c.size > 0 else 0)
            wavelet_features.append(np.std(c) if c.size > 0 else 0)
        wavelet_features = np.array(wavelet_features[:10])

        features = np.concatenate([mfccs, time_features, wavelet_features])
        if features.shape[0] != 25:
            raise ValueError(f"Expected 25 features, got {features.shape[0]} from {file_path}")
        return features

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        label = self.labels[idx]
        features = self.extract_features_from_path(file_path)
        return torch.tensor(features, dtype=torch.float32), torch.tensor(label)


# ======================================
# 2. LOAD DATASET & PRECOMPUTE FEATURES
# ======================================

root_dir = "/kaggle/input/qmsat-dataset/SMSAT-generated-data/SMSAT-generated-data"

dataset = SMSATDataset(root_dir=root_dir, samples_per_class=200)

# Precompute all features once to speed up CV and baselines
all_features = []
all_labels = []
all_subjects = []

for i in range(len(dataset)):
    feat, lab = dataset[i]
    all_features.append(feat.numpy())
    all_labels.append(lab.item())
    all_subjects.append(dataset.subject_ids[i])

X = np.stack(all_features)          # shape [N, 25]
y = np.array(all_labels)           # shape [N]
groups = np.array(all_subjects)    # for GroupKFold

print("Total samples:", X.shape[0])


# ======================================
# 3. MODELS: LSTM & SVM BASELINE
# ======================================

class CalmnessAnalysisModel(nn.Module):
    def __init__(self, input_dim=25, hidden_dim=128, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc1 = nn.Linear(hidden_dim * 2, 64)
        self.fc2 = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = x.unsqueeze(1)  # (B, 1, 25)
        lstm_out, _ = self.lstm(x)
        x = lstm_out[:, -1, :]  # last time step
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)


def train_lstm_one_fold(X_train, y_train, X_val, y_val, num_epochs=20, batch_size=32):
    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                             torch.tensor(y_train, dtype=torch.long))
    val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                           torch.tensor(y_val, dtype=torch.long))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = CalmnessAnalysisModel().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

        # validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                preds = out.argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        val_acc = correct / total
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return best_val_acc, model


def train_svm_one_fold(X_train, y_train, X_val, y_val):
    clf = SVC(kernel="rbf", C=1.0, gamma="scale")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    return acc, clf


# ======================================
# 4. SUBJECT-WISE K-FOLD CROSS-VALIDATION
# ======================================

from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

lstm_fold_accs = []
svm_fold_accs = []
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n===== FOLD {fold}/3 (class-wise stratified) =====")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # LSTM fold
    lstm_acc, _ = train_lstm_one_fold(X_train, y_train, X_val, y_val,
                                      num_epochs=15, batch_size=32)
    lstm_fold_accs.append(lstm_acc)
    print(f"LSTM Fold {fold} Accuracy: {lstm_acc*100:.2f}%")

    # SVM baseline fold
    svm_acc, _ = train_svm_one_fold(X_train, y_train, X_val, y_val)
    svm_fold_accs.append(svm_acc)
    print(f"SVM Fold {fold} Accuracy:  {svm_acc*100:.2f}%")

    fold_results.append({
        "Fold": fold,
        "LSTM_Accuracy": lstm_acc,
        "SVM_Accuracy": svm_acc,
        "NumTrainSamples": len(train_idx),
        "NumValSamples": len(val_idx)
    })



# ======================================
# 5. SUMMARY: MEAN ± STD, SAVE TO EXCEL
# ======================================

lstm_mean = np.mean(lstm_fold_accs)
lstm_std = np.std(lstm_fold_accs)
svm_mean = np.mean(svm_fold_accs)
svm_std = np.std(svm_fold_accs)

print("\n========== CROSS-VALIDATION SUMMARY ==========")
print(f"LSTM Accuracy: {lstm_mean*100:.2f} ± {lstm_std*100:.2f} %")
print(f"SVM  Accuracy: {svm_mean*100:.2f} ± {svm_std*100:.2f} %")

fold_df = pd.DataFrame(fold_results)
summary_df = pd.DataFrame({
    "Model": ["LSTM", "SVM"],
    "Mean_Accuracy": [lstm_mean, svm_mean],
    "Std_Accuracy": [lstm_std, svm_std]
})

with pd.ExcelWriter("/kaggle/working/cv_results_smsat.xlsx") as writer:
    fold_df.to_excel(writer, sheet_name="Per-Fold Results", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print("Cross-validation results saved to /kaggle/working/cv_results_smsat.xlsx")


# ======================================
# 6. TRAIN FINAL LSTM ON FULL DATA (FOR SHAP)
# ======================================

full_ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                        torch.tensor(y, dtype=torch.long))

full_loader = DataLoader(full_ds, batch_size=32, shuffle=True)

final_model = CalmnessAnalysisModel().to(device)
optimizer = optim.Adam(final_model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()

epochs_final = 10
for epoch in range(epochs_final):
    final_model.train()
    epoch_loss, correct, total = 0.0, 0, 0
    for xb, yb in full_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = final_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        preds = out.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)
    print(f"Final Training Epoch {epoch+1}/{epochs_final} | Loss={epoch_loss:.4f} | Acc={100*correct/total:.2f}%")

torch.save(final_model.state_dict(), "/kaggle/working/best_smsat_model_final_for_shap.pth")


# ======================================
# 7. SHAP EXPLAINABILITY ON FINAL MODEL
# ======================================

# Feature names: 13 MFCC + 2 time + 10 wavelet = 25
feature_names = [f"MFCC_{i}" for i in range(1, 14)] + ["ZCR", "RMS"] + \
                [f"Wavelet_{i}" for i in range(1, 11)]

# Background (for KernelExplainer) and samples to explain
n_background = min(20, X.shape[0])
n_explain = min(50, X.shape[0])

background_idx = np.random.choice(X.shape[0], size=n_background, replace=False)
explain_idx = np.random.choice(X.shape[0], size=n_explain, replace=False)

background = X[background_idx]
X_explain = X[explain_idx]

def model_prob(np_batch):
    """Wrapper: returns class probabilities for SHAP."""
    with torch.no_grad():
        t = torch.tensor(np_batch, dtype=torch.float32).to(device)
        logits = final_model(t)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    return probs

print("\nRunning SHAP KernelExplainer (this can take a bit)...")
explainer = shap.KernelExplainer(model_prob, background)
shap_values = explainer.shap_values(X_explain)  # list of [n_samples, n_features] per class

# SHAP summary plot for the predicted class (or choose a specific class index)
plt.figure()
shap.summary_plot(shap_values, X_explain,
                  feature_names=feature_names,
                  show=False)
plt.tight_layout()
plt.savefig("/kaggle/working/shap_summary_beeswarm.png", dpi=300)
plt.close()

# SHAP bar plot (mean |SHAP|)
plt.figure()
shap.summary_plot(shap_values, X_explain,
                  feature_names=feature_names,
                  plot_type="bar",
                  show=False)
plt.tight_layout()
plt.savefig("/kaggle/working/shap_summary_bar.png", dpi=300)
plt.close()

print("SHAP plots saved as:")
print("  /kaggle/working/shap_summary_beeswarm.png")
print("  /kaggle/working/shap_summary_bar.png")


Using device: cuda
Class counts: Counter({0: 200, 1: 200, 2: 200})
Total samples: 600

===== FOLD 1/3 (class-wise stratified) =====
LSTM Fold 1 Accuracy: 100.00%
SVM Fold 1 Accuracy:  100.00%

===== FOLD 2/3 (class-wise stratified) =====
LSTM Fold 2 Accuracy: 100.00%
SVM Fold 2 Accuracy:  100.00%

===== FOLD 3/3 (class-wise stratified) =====
LSTM Fold 3 Accuracy: 100.00%
SVM Fold 3 Accuracy:  100.00%

========== CROSS-VALIDATION SUMMARY ==========
LSTM Accuracy: 100.00 ± 0.00 %
SVM  Accuracy: 100.00 ± 0.00 %
Cross-validation results saved to /kaggle/working/cv_results_smsat.xlsx
Final Training Epoch 1/10 | Loss=7.4177 | Acc=92.33%
Final Training Epoch 2/10 | Loss=0.0780 | Acc=100.00%
Final Training Epoch 3/10 | Loss=0.0135 | Acc=100.00%
Final Training Epoch 4/10 | Loss=0.0055 | Acc=100.00%
Final Training Epoch 5/10 | Loss=0.0124 | Acc=100.00%
Final Training Epoch 6/10 | Loss=0.0025 | Acc=100.00%
Final Training Epoch 7/10 | Loss=0.0057 | Acc=100.00%
Final Training Epoch 8/10 | Loss=0.00

  0%|          | 0/50 [00:00<?, ?it/s]

SHAP plots saved as:
  /kaggle/working/shap_summary_beeswarm.png
  /kaggle/working/shap_summary_bar.png


50 & wave2vec

In [12]:
# ======================================
# SMSAT pipeline with wav2vec, 5-fold CLASS-WISE CV,
# baselines, CAM, per-fold Confusion & ROC, and SHAP for wav2vec CAM
# ======================================

import os
import random
import copy
import time

import numpy as np
import pandas as pd
import librosa
import pywt
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from collections import Counter

from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

from transformers import Wav2Vec2Processor, Wav2Vec2Model

import shap  # SHAP for explainability

sns.set_style("whitegrid")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ======================================
# 1. DATA INDEXING & FEATURE EXTRACTION
# ======================================

class SMSATAudioIndex:
    """
    Indexes files & labels (class-wise CV; no subject IDs).
    root/
      Music/Generated-data/*.wav
      Normal(Silence)/Generated-data/*.wav
      SpiritualMeditation/Generated-data/*.wav
    """
    def __init__(self, root_dir, samples_per_class=200):
        self.root_dir = root_dir
        self.file_paths = []
        self.labels = []
        self.class_counts = Counter()

        self.class_mapping = {
            "Music": 0,
            "Normal(Silence)": 1,
            "SpiritualMeditation": 2
        }

        for category, label in self.class_mapping.items():
            gen_path = os.path.join(root_dir, category, "Generated-data")
            if not os.path.isdir(gen_path):
                print(f"Warning: Missing folder {gen_path}")
                continue

            wav_files = [f for f in os.listdir(gen_path) if f.endswith(".wav")]

            if len(wav_files) > samples_per_class:
                wav_files = random.sample(wav_files, samples_per_class)

            for fname in wav_files:
                full_path = os.path.join(gen_path, fname)
                self.file_paths.append(full_path)
                self.labels.append(label)
                self.class_counts[label] += 1

        print("Class counts:", self.class_counts)
        print("Total files indexed:", len(self.file_paths))


def extract_handcrafted_features(file_path, sr_target=16000):
    """25-D: 13 MFCC, ZCR, RMS, 10 wavelet stats."""
    waveform, sr = librosa.load(file_path, sr=sr_target)

    mfccs = librosa.feature.mfcc(y=waveform, sr=sr, n_mfcc=13).mean(axis=1)

    zcr = librosa.feature.zero_crossing_rate(waveform)[0].mean()
    rms = librosa.feature.rms(y=waveform)[0].mean()
    time_features = np.array([zcr, rms])

    coeffs = pywt.wavedec(waveform, 'db4', level=4)
    wavelet_features = []
    for c in coeffs:
        wavelet_features.append(np.mean(c))
        wavelet_features.append(np.std(c))
    wavelet_features = np.array(wavelet_features[:10])

    features = np.concatenate([mfccs, time_features, wavelet_features])
    if features.shape[0] != 25:
        raise ValueError(f"Expected 25 features, got {features.shape[0]} from {file_path}")
    return features


# ======================================
# 2. LOAD INDEX & PRECOMPUTE FEATURES
# ======================================

root_dir = "/kaggle/input/qmsat-dataset/SMSAT-generated-data/SMSAT-generated-data"

index = SMSATAudioIndex(root_dir=root_dir, samples_per_class=50)

file_paths = index.file_paths
labels = np.array(index.labels)
y = labels
classes = np.unique(y)
num_classes = len(classes)

print("Precomputing handcrafted features...")
X_hand = np.stack([extract_handcrafted_features(p) for p in file_paths])
print("Handcrafted feature matrix shape:", X_hand.shape)

# ======================================
# 3. WAV2VEC2-BASE & EMBEDDINGS
# ======================================

print("\nLoading wav2vec2-base (facebook/wav2vec2-base)...")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(device)
wav2vec_model.eval()

def extract_wav2vec_embedding(file_path):
    waveform, sr = librosa.load(file_path, sr=16000)
    inputs = processor(waveform, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        out = wav2vec_model(inputs.input_values.to(device))
    hidden = out.last_hidden_state[0]  # [T, 768]
    emb = hidden.mean(dim=0).cpu().numpy()
    return emb

print("Precomputing wav2vec embeddings (this may take some time)...")
X_wav2vec = np.stack([extract_wav2vec_embedding(p) for p in file_paths])
print("Wav2Vec embedding matrix shape:", X_wav2vec.shape)

# ======================================
# 4. MODELS: LSTM, SVM, Wav2Vec Linear
# ======================================

class CalmnessAnalysisModel(nn.Module):
    """LSTM on 25-D handcrafted features."""
    def __init__(self, input_dim=25, hidden_dim=128, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc1 = nn.Linear(hidden_dim * 2, 64)
        self.fc2 = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = x.unsqueeze(1)  # (B, 1, 25)
        lstm_out, _ = self.lstm(x)
        x = lstm_out[:, -1, :]
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)


class Wav2VecLinearHead(nn.Module):
    """Linear classifier on top of 768-D wav2vec embeddings."""
    def __init__(self, in_dim=768, num_classes=3):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_classes)

    def forward(self, x):
        return self.fc(x)


def train_lstm_one_fold(X_train, y_train, X_val, y_val,
                        num_epochs=12, batch_size=32):
    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                             torch.tensor(y_train, dtype=torch.long))
    val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                           torch.tensor(y_val, dtype=torch.long))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = CalmnessAnalysisModel().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        # simple early selection by val acc
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                preds = logits.argmax(1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        acc = correct / total
        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return best_acc, model


def train_svm_one_fold(X_train, y_train, X_val, y_val):
    clf = SVC(kernel="rbf", C=1.0, gamma="scale")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    return acc, clf


def train_wav2vec_linear_one_fold(X_train, y_train, X_val, y_val,
                                  num_epochs=12, batch_size=32):
    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                             torch.tensor(y_train, dtype=torch.long))
    val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                           torch.tensor(y_val, dtype=torch.long))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = Wav2VecLinearHead(in_dim=X_train.shape[1], num_classes=num_classes).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                preds = logits.argmax(1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        acc = correct / total
        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return best_acc, model


# ======================================
# 5. CLASS-WISE STRATIFIED 5-FOLD CV + CONFUSION & ROC FOR wav2vec
# ======================================

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lstm_accs, svm_accs, wav_accs = [], [], []
fold_rows = []

os.makedirs("/kaggle/working/confusion_mats", exist_ok=True)
os.makedirs("/kaggle/working/roc_curves", exist_ok=True)

fold = 0
for train_idx, val_idx in skf.split(X_hand, y):
    fold += 1
    print(f"\n===== FOLD {fold}/5 (class-wise stratified) =====")

    Xh_train, Xh_val = X_hand[train_idx], X_hand[val_idx]
    Xw_train, Xw_val = X_wav2vec[train_idx], X_wav2vec[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # ---- LSTM on handcrafted ----
    lstm_acc, _ = train_lstm_one_fold(Xh_train, y_train, Xh_val, y_val)
    print(f"LSTM (25-feat) Acc: {lstm_acc*100:.2f}%")

    # ---- SVM on handcrafted ----
    svm_acc, _ = train_svm_one_fold(Xh_train, y_train, Xh_val, y_val)
    print(f"SVM  (25-feat) Acc: {svm_acc*100:.2f}%")

    # ---- Wav2Vec Linear ----
    wav_acc, wav_model = train_wav2vec_linear_one_fold(Xw_train, y_train, Xw_val, y_val)
    print(f"Wav2Vec Linear Acc: {wav_acc*100:.2f}%")

    lstm_accs.append(lstm_acc)
    svm_accs.append(svm_acc)
    wav_accs.append(wav_acc)

    # ---- Confusion Matrix & ROC (for wav2vec model) ----
    with torch.no_grad():
        logits_val = wav_model(torch.tensor(Xw_val, dtype=torch.float32).to(device))
        probs_val = torch.softmax(logits_val, dim=1).cpu().numpy()
        preds_val = probs_val.argmax(axis=1)

    # Confusion matrix
    cm = confusion_matrix(y_val, preds_val, labels=classes)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Wav2Vec Linear Confusion Matrix - Fold {fold}")
    cm_path = f"/kaggle/working/confusion_mats/confmat_wav2vec_fold{fold}.png"
    plt.tight_layout()
    plt.savefig(cm_path, dpi=300)
    plt.close()
    print(f"Saved confusion matrix for fold {fold} to {cm_path}")

    # ROC curves (one-vs-rest, micro-average)
    y_val_bin = label_binarize(y_val, classes=classes)
    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for i, c in enumerate(classes):
        fpr[c], tpr[c], _ = roc_curve(y_val_bin[:, i], probs_val[:, i])
        roc_auc[c] = auc(fpr[c], tpr[c])

    # Micro-average
    fpr["micro"], tpr["micro"], _ = roc_curve(y_val_bin.ravel(), probs_val.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    plt.figure(figsize=(6, 5))
    # Plot micro-average
    plt.plot(fpr["micro"], tpr["micro"],
             label=f"micro-average ROC (AUC = {roc_auc['micro']:.2f})")
    # Optionally: one curve per class
    for c in classes:
        plt.plot(fpr[c], tpr[c], linestyle="--", label=f"class {c} (AUC = {roc_auc[c]:.2f})")

    plt.plot([0, 1], [0, 1], "k--", label="Chance")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"Wav2Vec Linear ROC - Fold {fold}")
    plt.legend(loc="lower right", fontsize=8)
    roc_path = f"/kaggle/working/roc_curves/roc_wav2vec_fold{fold}.png"
    plt.tight_layout()
    plt.savefig(roc_path, dpi=300)
    plt.close()
    print(f"Saved ROC curve for fold {fold} to {roc_path}")

    fold_rows.append({
        "Fold": fold,
        "LSTM_Accuracy": lstm_acc,
        "SVM_Accuracy": svm_acc,
        "Wav2Vec_Linear_Accuracy": wav_acc,
        "NumTrain": len(train_idx),
        "NumVal": len(val_idx)
    })


# ======================================
# 6. SUMMARY STATISTICS & SAVE TO EXCEL
# ======================================

lstm_mean, lstm_std = np.mean(lstm_accs), np.std(lstm_accs)
svm_mean, svm_std = np.mean(svm_accs), np.std(svm_accs)
wav_mean, wav_std = np.mean(wav_accs), np.std(wav_accs)

print("\n========== CROSS-VALIDATION SUMMARY ==========")
print(f"LSTM (25-feat)      : {lstm_mean*100:.2f} ± {lstm_std*100:.2f} %")
print(f"SVM  (25-feat)      : {svm_mean*100:.2f} ± {svm_std*100:.2f} %")
print(f"Wav2Vec Linear (768): {wav_mean*100:.2f} ± {wav_std*100:.2f} %")

fold_df = pd.DataFrame(fold_rows)
summary_df = pd.DataFrame({
    "Model": ["LSTM_25feat", "SVM_25feat", "Wav2Vec_Linear"],
    "Mean_Accuracy": [lstm_mean, svm_mean, wav_mean],
    "Std_Accuracy": [lstm_std, svm_std, wav_std]
})

excel_path = "/kaggle/working/cv_results_smsat_with_wav2vec.xlsx"
with pd.ExcelWriter(excel_path) as writer:
    fold_df.to_excel(writer, sheet_name="Per-Fold Results", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print(f"Results saved to {excel_path}")


# ======================================
# 7. TRAIN FINAL Wav2Vec Linear (CAM + SHAP MODEL)
# ======================================

print("\nTraining final wav2vec linear classifier on all data (for CAM + SHAP)...")

full_wav_ds = TensorDataset(torch.tensor(X_wav2vec, dtype=torch.float32),
                            torch.tensor(y, dtype=torch.long))
full_loader = DataLoader(full_wav_ds, batch_size=32, shuffle=True)

wav_head = Wav2VecLinearHead(in_dim=X_wav2vec.shape[1], num_classes=num_classes).to(device)
optimizer = optim.Adam(wav_head.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(15):
    wav_head.train()
    epoch_loss, correct, total = 0.0, 0, 0
    for xb, yb in full_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = wav_head(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        preds = logits.argmax(1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)
    print(f"Final Wav2Vec Epoch {epoch+1}/15 | Loss={epoch_loss:.4f} | Acc={100*correct/total:.2f}%")

torch.save(wav_head.state_dict(), "/kaggle/working/wav2vec_linear_head_final.pth")
print("Final wav2vec linear head saved.")


# ======================================
# 8. CAM-LIKE MAPS (same wav2vec + linear model)
# ======================================

class_id_to_name = {0: "Music", 1: "Normal(Silence)", 2: "SpiritualMeditation"}

def compute_wav2vec_cam_for_file(file_path, target_class=None):
    waveform, sr = librosa.load(file_path, sr=16000)
    inputs = processor(waveform, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        outputs = wav2vec_model(inputs.input_values.to(device))
    hidden = outputs.last_hidden_state[0]  # [T, 768]

    with torch.no_grad():
        emb = hidden.mean(dim=0, keepdim=True)
        logits = wav_head(emb.to(device))
        probs = torch.softmax(logits, dim=1).cpu().numpy().squeeze()
        pred_class = int(np.argmax(probs))

    if target_class is None:
        target_class = pred_class

    W = wav_head.fc.weight.detach().cpu().numpy()  # [C, 768]
    w_c = W[target_class]  # [768]

    A = hidden.detach().cpu().numpy()  # [T, 768]
    cam = A @ w_c  # [T]
    cam = np.maximum(cam, 0)
    if cam.max() > 0:
        cam = cam / cam.max()

    # upsample to waveform
    T = cam.shape[0]
    L = waveform.shape[0]
    x_old = np.linspace(0, 1, T)
    x_new = np.linspace(0, 1, L)
    cam_upsampled = np.interp(x_new, x_old, cam)

    return waveform, sr, cam_upsampled, pred_class, probs

os.makedirs("/kaggle/working/cam_plots", exist_ok=True)

sample_per_class = {}
for path, lab in zip(file_paths, y):
    cname = class_id_to_name[lab]
    if cname not in sample_per_class:
        sample_per_class[cname] = path
    if len(sample_per_class) == num_classes:
        break

for cname, path in sample_per_class.items():
    print(f"\nGenerating CAM for class {cname}, file: {os.path.basename(path)}")
    waveform, sr, cam_signal, pred_class, probs = compute_wav2vec_cam_for_file(path)

    t = np.arange(len(waveform)) / sr

    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(t, waveform, alpha=0.7)
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Amplitude")
    ax1.set_title(f"Wav2Vec CAM - True: {cname}, Pred: {class_id_to_name[pred_class]}")

    ax2 = ax1.twinx()
    ax2.plot(t, cam_signal, alpha=0.7, color="red")
    ax2.set_ylabel("CAM Importance", color="red")
    ax2.tick_params(axis="y", labelcolor="red")

    plt.tight_layout()
    out_path = f"/kaggle/working/cam_plots/wav2vec_cam_{cname}.png"
    plt.savefig(out_path, dpi=300)
    plt.close()
    print(f"CAM plot saved to {out_path}")

print("\nAll CAM-like wav2vec maps generated and saved in /kaggle/working/cam_plots/")


# ======================================
# 9. SHAP EXPLAINABILITY (ONLY FOR wav2vec CAM MODEL)
# ======================================

print("\nRunning SHAP on wav2vec linear head (CAM model)...")

# Use a small background and explanation set for speed
n_background = min(30, X_wav2vec.shape[0])
n_explain = min(80, X_wav2vec.shape[0])

bg_idx = np.random.choice(X_wav2vec.shape[0], size=n_background, replace=False)
ex_idx = np.random.choice(X_wav2vec.shape[0], size=n_explain, replace=False)

background = torch.tensor(X_wav2vec[bg_idx], dtype=torch.float32).to(device)
X_explain = torch.tensor(X_wav2vec[ex_idx], dtype=torch.float32).to(device)

def wav_head_prob(x_np):
    with torch.no_grad():
        x_t = torch.tensor(x_np, dtype=torch.float32).to(device)
        logits = wav_head(x_t)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    return probs

# KernelExplainer is model-agnostic; DeepExplainer would also work,
# but KernelExplainer is clearer here.
explainer = shap.KernelExplainer(wav_head_prob, background.cpu().numpy())
shap_values = explainer.shap_values(X_explain.cpu().numpy())

os.makedirs("/kaggle/working/shap_plots", exist_ok=True)

# Due to 768 dims, this will be dense, but reviewers just need proof-of-concept.
# Summary beeswarm
plt.figure()
shap.summary_plot(shap_values, X_explain.cpu().numpy(),
                  show=False)
plt.tight_layout()
plt.savefig("/kaggle/working/shap_plots/shap_summary_beeswarm_wav2vec.png", dpi=300)
plt.close()

# Summary bar (mean |SHAP|)
plt.figure()
shap.summary_plot(shap_values, X_explain.cpu().numpy(),
                  plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("/kaggle/working/shap_plots/shap_summary_bar_wav2vec.png", dpi=300)
plt.close()

print("SHAP plots saved in /kaggle/working/shap_plots/")


Using device: cuda
Class counts: Counter({0: 50, 1: 50, 2: 50})
Total files indexed: 150
Precomputing handcrafted features...


KeyboardInterrupt: 

In [8]:
!zip -r output.zip /kaggle/working/

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/cam_plots/ (stored 0%)
  adding: kaggle/working/cam_plots/wav2vec_cam_Music.png (deflated 3%)
  adding: kaggle/working/cam_plots/wav2vec_cam_SpiritualMeditation.png (deflated 3%)
  adding: kaggle/working/cam_plots/wav2vec_cam_Normal(Silence).png (deflated 2%)
  adding: kaggle/working/shap_plots/ (stored 0%)
  adding: kaggle/working/shap_plots/shap_summary_bar_wav2vec.png (deflated 27%)
  adding: kaggle/working/shap_plots/shap_summary_beeswarm_wav2vec.png (deflated 27%)
  adding: kaggle/working/wav2vec_linear_head_final.pth (deflated 15%)
  adding: kaggle/working/roc_curves/ (stored 0%)
  adding: kaggle/working/roc_curves/roc_wav2vec_fold5.png (deflated 18%)
  adding: kaggle/working/roc_curves/roc_wav2vec_fold3.png (deflated 18%)
  adding: kaggle/working/roc_curves/roc_wav2vec_fold2.png (deflated 18%)
  adding: kaggle/working/roc_curves/roc_wav2vec_fold1.png (deflated 18%)
  adding: kaggle/working/roc_curves/roc_wav2vec_fold